In [11]:
# 📋 規格書

學號：411312189 姓名：王冠元 卷別：A

## 1. 系統目的（一句話）
透過 AI 解析使用者的一句話來自動處理系學會活動的報名、取消與剩餘名額查詢，並提供各活動報名統計。

## 2. 使用者會怎麼說（至少 5 句）
1. 我想報名迎新晚會
2. 許小明要取消程式工作坊
3. 查詢籃球友誼賽剩餘名額
4. 王小美要報名迎新晚會和兩個朋友
5. 各活動報名狀況

## 3. 資料表（2 張）
| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| activities | id | INTEGER PRIMARY KEY AUTOINCREMENT | 活動編號 |
| activities | name | TEXT | 活動名稱 |
| activities | date | TEXT | 活動日期 |
| activities | quota | INTEGER | 活動總名額 |
| registrations | id | INTEGER PRIMARY KEY AUTOINCREMENT | 報名紀錄編號 |
| registrations | student_name | TEXT | 學生姓名 |
| registrations | activity_name | TEXT | 活動名稱 |
| registrations | count | INTEGER | 報名人數（含攜伴） |
| registrations | status | TEXT | 狀態（已報名 / 已取消） |

初始資料：
- 迎新晚會 | 2026-11-20 | 6
- 程式工作坊 | 2026-11-25 | 4
- 籃球友誼賽 | 2026-11-28 | 3

## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：`動作,活動,人數`
- 動作只能是：報名、取消、查詢、統計、未知
- 活動只能是：迎新晚會、程式工作坊、籃球友誼賽、未知
- 人數的規則：若有提到攜伴或人數請萃取數字，未提及則預設為 1 人
- 無法判斷時輸出：`未知,未知,1`
- AI 不可以做的事：不可以自行修改資料庫或捏造題目未提供的活動

## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 報名 | 名額足夠且未超過個人上限（最多同時報名 2 個活動）、未重複報名 | 報名成功訊息 | 新增一筆記錄，狀態為「已報名」 |
| 取消 | 該學生有此活動的有效報名紀錄 | 取消成功訊息，釋出名額 | 將該筆記錄狀態改為「已取消」 |
| 查詢 | 活動名稱正確 | 顯示該活動的剩餘名額 | 無變化 |
| 統計查詢 | 使用者詢問各活動報名狀況 | 列表顯示各活動日期、名額、已報名人數與剩餘名額 | 查詢並聚合資料庫資料 |

## 6. 例外情況（至少 3 個）
1. 報名人數超過 3 人或小於 1 人時拒絕並提示。
2. 重複報名同一活動時拒絕並提示。
3. 查詢或取消時找不到對應的活動或學生紀錄時給予提示。

## 7. 測試清單
**7-1 AI 解讀測試（8～10 句）**
| # | 類型 | 使用者說的話 | 預期動作 | 預期活動 | 預期人數 |
|---|---|---|---|---|---|
| 1 | 正常 | 我想報名迎新晚會 | 報名 | 迎新晚會 | 1 |
| 2 | 正常 | 王小美要報名程式工作坊和兩個朋友 | 報名 | 程式工作坊 | 3 |
| 3 | 正常 | 取消籃球友誼賽 | 取消 | 籃球友誼賽 | 1 |
| 4 | 正常 | 查詢迎新晚會剩餘名額 | 查詢 | 迎新晚會 | 1 |
| 5 | 正常 | 各活動報名狀況 | 統計 | 未知 | 1 |
| 6 | 邊界 | 報名程式工作坊四個人 | 報名 | 程式工作坊 | 4 |
| 7 | 錯誤 | 明天一起去吃火鍋 | 未知 | 未知 | 1 |
| 8 | 邊界 | 查詢籃球比賽 | 查詢 | 籃球友誼賽 | 1 |

**7-2 流程測試（8～10 步，依序執行）**
| # | 姓名 | 使用者說的話 | 預期結果 |
|---|---|---|---|
| 1 | 王小明 | 我想報名迎新晚會 | 報名成功 |
| 2 | 王小明 | 我想報名程式工作坊 | 報名成功 |
| 3 | 王小明 | 我想報名籃球友誼賽 | 拒絕（超過每人同時報名 2 個活動上限） |
| 4 | 李小華 | 報名迎新晚會和兩個朋友 | 報名成功（佔 3 人名額） |
| 5 | 王小明 | 取消迎新晚會 | 取消成功，釋出名額 |
| 6 | 王小明 | 我想報名籃球友誼賽 | 報名成功 |
| 7 | 張小麗 | 查詢迎新晚會剩餘名額 | 顯示正確剩餘名額 |
| 8 | 管理員 | 各活動報名狀況 | 列表顯示所有活動的詳細統計數據 |

SyntaxError: invalid character '：' (U+FF1A) (703967366.py, line 3)

In [12]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型（請勿修改）
# ===========================================================
!pip install -q groq gradio

import os
import re
import time
import sqlite3
import pandas as pd
from IPython.display import display
from groq import Groq
import gradio as gr

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    try:
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except Exception:
        print("❌ 讀不到金鑰：請到左側 🔑 新增名稱為 GROQ_API_KEY 的密鑰")
        raise
else:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
    if not GROQ_API_KEY:
        raise RuntimeError("找不到環境變數 GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)
MODEL = "openai/gpt-oss-20b"
REASONING_EFFORT = "low"
print("✅ 環境：", "Colab" if IN_COLAB else "本地端", "｜模型：", MODEL)

✅ 環境： Colab ｜模型： openai/gpt-oss-20b


In [13]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================
import sqlite3

def init_db():
    conn = sqlite3.connect('activities.db')
    cursor = conn.cursor()
    try:
        cursor.execute("DROP TABLE IF EXISTS registrations")
        cursor.execute("DROP TABLE IF EXISTS activities")

        cursor.execute("""
            CREATE TABLE activities (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT UNIQUE,
                date TEXT,
                quota INTEGER
            )
        """)

        cursor.execute("""
            CREATE TABLE registrations (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                student_name TEXT,
                activity_name TEXT,
                count INTEGER,
                status TEXT
            )
        """)

        initial_activities = [
            ("迎新晚會", "2026-11-20", 6),
            ("程式工作坊", "2026-11-25", 4),
            ("籃球友誼賽", "2026-11-28", 3)
        ]
        cursor.executemany("""
            INSERT INTO activities (name, date, quota)
            VALUES (?, ?, ?)
        """, initial_activities)

        conn.commit()
        print("✅ 資料庫與初始資料建立成功！")
    finally:
        conn.close()

init_db()

✅ 資料庫與初始資料建立成功！


In [14]:
# ============================================================
# 第 3 格：顯示資料表
# 提醒：定義 show_tables() 並呼叫一次
# ============================================================
def show_tables():
    conn = sqlite3.connect('activities.db')
    try:
        print("--- 【activities 活動表】 ---")
        df_act = pd.read_sql_query("SELECT * FROM activities", conn)
        display(df_act)

        print("--- 【registrations 報名紀錄表】 ---")
        df_reg = pd.read_sql_query("SELECT * FROM registrations", conn)
        display(df_reg)
    finally:
        conn.close()

show_tables()

--- 【activities 活動表】 ---


,id,name,date,quota
0,1,迎新晚會,2026-11-20,6
1,2,程式工作坊,2026-11-25,4
2,3,籃球友誼賽,2026-11-28,3


--- 【registrations 報名紀錄表】 ---


,id,student_name,activity_name,count,status


In [15]:
# ============================================================
# 第 4 格：AI 解讀函式
# 提醒：回傳 (動作, 活動, 人數, 原始回覆)；不可直接相信 AI 的回覆
# ============================================================
ai_cache = {}

def parse_intent(user_input):
    if user_input in ai_cache:
        return ai_cache[user_input]

    prompt = f"""
請分析以下使用者針對系學會活動系統說的話，並萃取三個欄位：動作、活動、人數。
【動作選項】：報名、取消、查詢、統計、未知
【活動選項】：迎新晚會、程式工作坊、籃球友誼賽、未知
【人數規則】：若有提到攜伴或人數請萃取數字（例如「兩個朋友」代表 3 人），未提及則預設為 1。

使用者說的話：「{user_input}」

請嚴格按照以下格式回覆，不要有多餘的字：
動作,活動,人數
"""

    raw_response = "未知,未知,1"
    for attempt in range(3):
        try:
            time.sleep(2)
            completion = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                reasoning_effort=REASONING_EFFORT
            )
            raw_response = completion.choices[0].message.content.strip()
            break
        except Exception as e:
            print(f"⚠️ 呼叫 AI 發生錯誤 (第 {attempt+1} 次): {e}")
            time.sleep(30)

    parts = [p.strip() for p in raw_response.split(',')]
    action = parts[0] if len(parts) > 0 and parts[0] in ["報名", "取消", "查詢", "統計"] else "未知"
    activity = parts[1] if len(parts) > 1 and parts[1] in ["迎新晚會", "程式工作坊", "籃球友誼賽"] else "未知"

    count = 1
    if len(parts) > 2:
        nums = re.findall(r'\d+', parts[2])
        if nums:
            count = int(nums[0])

    result = (action, activity, count, raw_response)
    ai_cache[user_input] = result
    return result

In [16]:
# ============================================================
# 第 5 格：AI 解讀測試
# 提醒：8～10 句，要與規格書 7-1 一致
# ============================================================
test_cases = [
    ("我想報名迎新晚會", "報名", "迎新晚會", 1),
    ("王小美要報名程式工作坊和兩個朋友", "報名", "程式工作坊", 3),
    ("取消籃球友誼賽", "取消", "籃球友誼賽", 1),
    ("查詢迎新晚會剩餘名額", "查詢", "迎新晚會", 1),
    ("各活動報名狀況", "統計", "未知", 1),
    ("報名程式工作坊四個人", "報名", "程式工作坊", 4),
    ("明天一起去吃火鍋", "未知", "未知", 1),
    ("查詢籃球比賽", "查詢", "籃球友誼賽", 1)
]

passed_count = 0
for idx, (text, exp_act, exp_item, exp_cnt) in enumerate(test_cases, 1):
    act, item, cnt, raw = parse_intent(text)
    match = (act == exp_act) and (item == exp_item) and (cnt == exp_cnt)
    if match:
        passed_count += 1
    print(f"測試 {idx}: 「{text}」")
    print(f"  -> AI原始: {raw}")
    print(f"  -> 系統解讀: 動作={act}, 活動={item}, 人數={cnt} | 結果: {'✅ 通過' if match else '❌ 失敗'}\n")

print(f"🎯 AI 解讀測試完成！共通過 {passed_count} / {len(test_cases)} 句。")

測試 1: 「我想報名迎新晚會」
  -> AI原始: 報名,迎新晚會,1
  -> 系統解讀: 動作=報名, 活動=迎新晚會, 人數=1 | 結果: ✅ 通過

測試 2: 「王小美要報名程式工作坊和兩個朋友」
  -> AI原始: 報名,程式工作坊,3
  -> 系統解讀: 動作=報名, 活動=程式工作坊, 人數=3 | 結果: ✅ 通過

測試 3: 「取消籃球友誼賽」
  -> AI原始: 取消,籃球友誼賽,1
  -> 系統解讀: 動作=取消, 活動=籃球友誼賽, 人數=1 | 結果: ✅ 通過

測試 4: 「查詢迎新晚會剩餘名額」
  -> AI原始: 查詢,迎新晚會,1
  -> 系統解讀: 動作=查詢, 活動=迎新晚會, 人數=1 | 結果: ✅ 通過

測試 5: 「各活動報名狀況」
  -> AI原始: 查詢,未知,1
  -> 系統解讀: 動作=查詢, 活動=未知, 人數=1 | 結果: ❌ 失敗

測試 6: 「報名程式工作坊四個人」
  -> AI原始: 報名,程式工作坊,4
  -> 系統解讀: 動作=報名, 活動=程式工作坊, 人數=4 | 結果: ✅ 通過

測試 7: 「明天一起去吃火鍋」
  -> AI原始: 動作,活動,人數
未知,未知,1
  -> 系統解讀: 動作=未知, 活動=未知, 人數=1 | 結果: ✅ 通過

測試 8: 「查詢籃球比賽」
  -> AI原始: 查詢,籃球友誼賽,1
  -> 系統解讀: 動作=查詢, 活動=籃球友誼賽, 人數=1 | 結果: ✅ 通過

🎯 AI 解讀測試完成！共通過 7 / 8 句。


In [17]:
# ============================================================
# 第 6 格：功能函式
# 提醒：fetchone() 後先檢查 None；用 try/finally 關閉連線；統計用 JOIN＋GROUP BY
# ============================================================
def process_request(student_name, user_input):
    if not student_name and "統計" not in user_input and "查詢" not in user_input:
        return "❌ 錯誤：請先輸入學生姓名才能進行報名或取消！"

    action, activity_name, count, _ = parse_intent(user_input)

    if action == "未知":
        return "🤖 AI 聽不太懂您的意思。目前的活動有：迎新晚會、程式工作坊、籃球友誼賽。"

    conn = sqlite3.connect('activities.db')
    cursor = conn.cursor()
    try:
        if action == "統計":
            query = """
                SELECT a.name AS 活動名稱, a.date AS 日期, a.quota AS 總名額,
                       IFNULL(SUM(CASE WHEN r.status = '已報名' THEN r.count ELSE 0 END), 0) AS 已報名人數,
                       (a.quota - IFNULL(SUM(CASE WHEN r.status = '已報名' THEN r.count ELSE 0 END), 0)) AS 剩餘名額
                FROM activities a
                LEFT JOIN registrations r ON a.name = r.activity_name
                GROUP BY a.name, a.date, a.quota
            """
            df = pd.read_sql_query(query, conn)
            return df.to_string(index=False)

        cursor.execute("SELECT quota FROM activities WHERE name = ?", (activity_name,))
        row = cursor.fetchone()
        if row is None:
            return f"❌ 找不到活動：{activity_name}"
        total_quota = row[0]

        if action == "查詢":
            cursor.execute("""
                SELECT IFNULL(SUM(CASE WHEN status = '已報名' THEN count ELSE 0 END), 0)
                FROM registrations WHERE activity_name = ?
            """, (activity_name,))
            reg_row = cursor.fetchone()
            enrolled = reg_row[0] if reg_row else 0
            left = total_quota - enrolled
            return f"📅 活動【{activity_name}】總名額 {total_quota} 人，目前已報名 {enrolled} 人，剩餘名額 {left} 人。"

        if action == "報名":
            if count < 1 or count > 3:
                return f"❌ 報名人數 {count} 人不符合規定（每筆報名人數必須在 1～3 人之間）。"

            cursor.execute("SELECT status FROM registrations WHERE student_name = ? AND activity_name = ?", (student_name, activity_name))
            dup_row = cursor.fetchone()
            if dup_row and dup_row[0] == '已報名':
                return f"❌ {student_name} 已經報名過【{activity_name}】了，不可重複報名！"

            cursor.execute("SELECT COUNT(*) FROM registrations WHERE student_name = ? AND status = '已報名'", (student_name,))
            limit_row = cursor.fetchone()
            current_active_count = limit_row[0] if limit_row else 0
            if current_active_count >= 2:
                return f"❌ {student_name} 目前已報名 {current_active_count} 個活動，已達每人同時報名 2 個活動的上限！"

            cursor.execute("""
                SELECT IFNULL(SUM(CASE WHEN status = '已報名' THEN count ELSE 0 END), 0)
                FROM registrations WHERE activity_name = ?
            """, (activity_name,))
            enrolled_row = cursor.fetchone()
            enrolled = enrolled_row[0] if enrolled_row else 0

            if (enrolled + count) > total_quota:
                return f"❌ 報名失敗！【{activity_name}】剩餘名額不足（目前剩餘 {total_quota - enrolled} 人，您欲報名 {count} 人）。"

            cursor.execute("""
                INSERT INTO registrations (student_name, activity_name, count, status)
                VALUES (?, ?, ?, '已報名')
            """, (student_name, activity_name, count))
            conn.commit()
            return f"✅ {student_name} 成功報名【{activity_name}】（人數：{count} 人）！"

        if action == "取消":
            cursor.execute("SELECT id, status FROM registrations WHERE student_name = ? AND activity_name = ? AND status = '已報名'", (student_name, activity_name))
            reg_rec = cursor.fetchone()
            if reg_rec is None:
                return f"❌ 找不到 {student_name} 在【{activity_name}】的有效報名紀錄，無法取消。"

            cursor.execute("UPDATE registrations SET status = '已取消' WHERE id = ?", (reg_rec[0],))
            conn.commit()
            return f"✅ 已成功為 {student_name} 取消【{activity_name}】的報名！"

    finally:
        conn.close()

In [18]:
# ============================================================
# 第 7 格：主流程與流程測試
# 提醒：流程測試 8～10 步；最後呼叫 show_tables()
# ============================================================
print("=== 開始執行流程測試 ===")
flow_tests = [
    ("王小明", "我想報名迎新晚會"),
    ("王小明", "我想報名程式工作坊"),
    ("王小明", "我想報名籃球友誼賽"),
    ("李小華", "報名迎新晚會和兩個朋友"),
    ("王小明", "取消迎新晚會"),
    ("王小明", "我想報名籃球友誼賽"),
    ("張小麗", "查詢迎新晚會剩餘名額"),
    ("管理員", "各活動報名狀況")
]

for idx, (name, text) in enumerate(flow_tests, 1):
    print(f"\n步驟 {idx} [姓名: {name}] 說：「{text}」")
    res = process_request(name, text)
    print(f"系統回應：\n{res}")

print("\n=== 測試結束，最終資料庫狀態 ===")
show_tables()

=== 開始執行流程測試 ===

步驟 1 [姓名: 王小明] 說：「我想報名迎新晚會」
系統回應：
✅ 王小明 成功報名【迎新晚會】（人數：1 人）！

步驟 2 [姓名: 王小明] 說：「我想報名程式工作坊」
系統回應：
✅ 王小明 成功報名【程式工作坊】（人數：1 人）！

步驟 3 [姓名: 王小明] 說：「我想報名籃球友誼賽」
系統回應：
❌ 王小明 目前已報名 2 個活動，已達每人同時報名 2 個活動的上限！

步驟 4 [姓名: 李小華] 說：「報名迎新晚會和兩個朋友」
系統回應：
🤖 AI 聽不太懂您的意思。目前的活動有：迎新晚會、程式工作坊、籃球友誼賽。

步驟 5 [姓名: 王小明] 說：「取消迎新晚會」
系統回應：
✅ 已成功為 王小明 取消【迎新晚會】的報名！

步驟 6 [姓名: 王小明] 說：「我想報名籃球友誼賽」
系統回應：
✅ 王小明 成功報名【籃球友誼賽】（人數：1 人）！

步驟 7 [姓名: 張小麗] 說：「查詢迎新晚會剩餘名額」
系統回應：
📅 活動【迎新晚會】總名額 6 人，目前已報名 0 人，剩餘名額 6 人。

步驟 8 [姓名: 管理員] 說：「各活動報名狀況」
系統回應：
❌ 找不到活動：未知

=== 測試結束，最終資料庫狀態 ===
--- 【activities 活動表】 ---


,id,name,date,quota
0,1,迎新晚會,2026-11-20,6
1,2,程式工作坊,2026-11-25,4
2,3,籃球友誼賽,2026-11-28,3


--- 【registrations 報名紀錄表】 ---


,id,student_name,activity_name,count,status
0,1,王小明,迎新晚會,1,已取消
1,2,王小明,程式工作坊,1,已報名
2,3,王小明,籃球友誼賽,1,已報名


In [19]:
# ============================================================
# 第 8 格：Gradio 網頁
# 提醒：if IN_COLAB: demo.launch(share=True)
# ============================================================
def gradio_chat(student_name, user_input):
    if not user_input.strip():
        return "請輸入您的需求訊息。"
    return process_request(student_name.strip(), user_input.strip())

demo = gr.Interface(
    fn=gradio_chat,
    inputs=[
        gr.Textbox(label="學生姓名（查詢或看統計可不填）", placeholder="例如：王小明"),
        gr.Textbox(label="您的需求一句話", placeholder="例如：我想報名迎新晚會 / 各活動報名狀況")
    ],
    outputs=gr.Textbox(label="系統回應"),
    title="系學會活動報名系統",
    description="請輸入姓名與一句話來進行活動報名、取消、查詢剩餘名額或檢視統計狀況。"
)

if __name__ == "__main__":
    if IN_COLAB:
        demo.launch(share=True)
    else:
        demo.launch()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://02c0d62a359d7d4a21.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
